This notebook must be executed inside the FINN container provided by Xilinx.

Follow the steps of the [FINN docks](https://finn.readthedocs.io/en/latest/), section [Quickstart](https://finn.readthedocs.io/en/latest/getting_started.html#running-finn-in-docker), to download, build and verify the container installation.

You must also move your project folder to inside the same folder the repository is located.

To start the container, go to the folder where the repo was installed and run $ ./run-docker.sh notebook

If you are using vscode, you can select the notebook kernel inside the container to run the code.

This notebook is strongly based on Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook and 0BAB1 [2_finn_hardware_layers.ipynb](https://github.com/0BAB1/tutorial-snippets/blob/main/8%20Python%20to%20FPGA/2_finn_hardware_layers.ipynb) notebook.

Check them out for deeper instructions.

# Setup Python Paths for Libraries

In [1]:
import os
import sys

# Correct the path where the environment starts to be the same folder of the notebook, so that the imports work correctly.
# When starting the container with the notebook server, the script hardcodes the Jupyter server to start inside the ./notebooks folder.
os.chdir('../QFast-SCNN_with_Brevitas_and_FINN/finn_environment')

# Add the train_environment directory to the system path to allow imports from there
sys.path.append(os.path.abspath('../train_environment'))
print(sys.path)

['/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/tmp/home_dir/.local/lib/python3.10/site-packages', '/home/jose-vitor/finn-repo/deps/qonnx/src', '/home/jose-vitor/finn-repo/deps/finn-experimental/src', '/home/jose-vitor/finn-repo/deps/brevitas/src', '/home/jose-vitor/finn-repo/deps/pyverilator', '/home/jose-vitor/finn-repo/src', '/usr/local/lib/python3.10/dist-packages', '/workspace/src/dataset-loading', '/usr/lib/python3/dist-packages', '/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment']


In [2]:
# Import all used libraries an functions and create all folders needed for the project, for convenience when is needed to run only a part of the project.
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.datasets import Cityscapes
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
from config import IM_SIZE, NUM_CLASSES, DATA_PATH, BIT_WIDTH
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.fold_constants import FoldConstants
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from onnx import helper
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph
from finn.transformation.streamline import Streamline
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveIdenticalOpPastJoinOp, MoveLinearPastEltwiseAdd, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

brevitas_onnx_path = f'./from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)

tidy_path = f'./tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)

preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)

postproc_path = f'./postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)

streamlined_path = f'./streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)

ready_for_hw_path = f'./ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)

hw_layers_path = f'./with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)

/home/jose-vitor/finn-repo/deps/brevitas/src/brevitas/__init__.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packag

# Convert Brevitas exported QONNX to FINN

ModelWrapper is a wrapper around the ONNX model which provides several helper functions to make it easier to work with the model.

ConvertQONNXtoFINN Convert the model to the FINN-ONNX model. The main difference of this ONNX from the standard is how the quantization is handled.

In [3]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.transformation.qonnx.infer_quant_avg_pool_2d import AvgPoolAndTruncToQuantAvgPool
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from pathlib import Path
from config import BIT_WIDTH

# Setup Path
onnx_path = f'../onnx/quant_model_{BIT_WIDTH}_bits.onnx'
onnx_name = Path(onnx_path).stem

# Note: The original tfc_end2end_example.ipynb does not run the InferShapes and InferDataTypes transformations, but without them the ConvertQONNXtoFINN transformation will fail with an error of missing shape information.
qonnx_cleanup(onnx_path, out_file=onnx_path)
model = ModelWrapper(onnx_path)
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(ConvertQONNXtoFINN())

brevitas_onnx_path = f'./from_brevitas_onnx/{onnx_name}_from_brevitas.onnx'
Path(brevitas_onnx_path).parent.mkdir(parents=True, exist_ok=True)
model.save(brevitas_onnx_path)

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in a future release.To silence this warning, please use `helper.tensor_dtype_to_np_dtype` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


In [4]:
showInNetron(brevitas_onnx_path)

Serving './from_brevitas_onnx/quant_model_8_bits_from_brevitas.onnx' at http://0.0.0.0:8081


# Setup Model for FINN

* Tidy up (and also after EACH step)
* Pre (data feed) / Post proc (top k)
* Model streamlining (Main step) + smaller example
* Model HW Layers (Generates Matrix Vector Activation Units for fc layers)
* Model data flow partitions (Generate a sub-graph for all HW convertible nodes)
* Specialize layer, ready for hw conversion (generates hls for the dataflow partition node)

### Tidy Up

In [5]:
from finn.util.visualization import showSrc, showInNetron
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.core.modelwrapper import ModelWrapper
from config import IM_SIZE

model = ModelWrapper(brevitas_onnx_path)

# TIDY UP
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
tidy_path = f'./tidy_onnx/{onnx_name}_tidy.onnx'
Path(tidy_path).parent.mkdir(parents=True, exist_ok=True)
model.save(tidy_path)

### Pre processing

FINN model expects UINT8 input. According to Xilinx, this is highly beneficial for performance, because you can directly input raw data to the model, instead of relying on CPU for pre processing.

The the QFast-SCNN model exported to QONNX has the pre processing layers integrated in the Pytorch model, so the expected input is already from 0 to 255.

If the target model was trained with tensor inputs different than [0, 255], like the standard torch.Tensor [0, 1] or tensors with Imagenet normalization, you have two main options to follow:
* Modify your Pytorch model only for the QONNX export, integrating the pre processing inside the model (the option I have chosen for QFast-SCNN).
* Add the preprocessing layers in the QONNX model, following the "Adding Pre- and Postprocessing" section of Xilinx [tfc_end2end_example.ipynb](https://github.com/Xilinx/finn/blob/main/notebooks/end2end_example/bnn-pynq/tfc_end2end_example.ipynb) notebook.

In [6]:
from finn.util.pytorch import ToTensor
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from qonnx.util.cleanup import cleanup as qonnx_cleanup
import torch
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from onnx import helper

# PRE PROC : NONE
model = ModelWrapper(tidy_path)

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

# verify if the sizes of first and last node outputs are correct
first_node_out = model.graph.node[0].output[0]
last_node_out = model.graph.node[-1].output[0]
print(f"\nOutput shape shape of first node ({model.graph.node[0].op_type}): {model.get_tensor_shape(first_node_out)}")
print(f"Output shape shape of last node ({model.graph.node[-1].op_type}): {model.get_tensor_shape(last_node_out)}")

# verify if input and output datatypes are correct
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")
print(f"Output datatype: {model.get_tensor_datatype(last_node_out)}")

# Save the preprocessed model
preproc_path = f'./preproc_onnx/{onnx_name}_preproc.onnx'
Path(preproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(preproc_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))


Output shape shape of first node (Mul): [1, 3, 1024, 1024]
Output shape shape of last node (Add): [1, 19, 128, 128]

Input datatype: UINT8
Output datatype: FLOAT32


In [8]:
showInNetron(preproc_path)

Serving './preproc_onnx/quant_model_8_bits_preproc.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Pytorch Model and QONNX Model

Optional but highly recommended step. The Pytorch model will be loaded with the "finn" mode so the test input of this model is the same as the QONNX model.

In [8]:
import torch
from torchvision import transforms
from torchvision.datasets import Cityscapes
from my_finn_utils import load_state_dict, generate_cityscapes_labels, IdToTrainIdTransform
import models.QFastSCNN as qfscnn
from config import NUM_CLASSES, DATA_PATH, BIT_WIDTH

lable_conversion, id_names = generate_cityscapes_labels()

# Defining the Cityscapes validation dataset.
val_dataset = Cityscapes(
    root=DATA_PATH,
    split='val',
    mode='fine',
    target_type='semantic',
    transform=transforms.PILToTensor(), # Converting the PIL images to tensors, keeping the original pixel values (0-255) which is important for the quantized model that expects UINT8 inputs.
    target_transform=transforms.Compose([
        transforms.PILToTensor(), # Converting the PIL masks to tensors, keeping the original pixel values (0-255).
        IdToTrainIdTransform(lable_conversion), # Converting the original Cityscapes labels to the 19 classes used for training and evaluation, as per the Cityscapes benchmark.
    ])
)

# Importing the test image and mask
img_tensor, smnt_tensor = val_dataset[0]
img_tensor = img_tensor.unsqueeze(0) # Add batch dimension
print("Input image shape:", img_tensor.shape)
print("Input image dtype:", img_tensor.dtype)
print("Input mask shape:", smnt_tensor.shape)
print("Input mask dtype:", smnt_tensor.dtype)

# Creating a Brevitas model instance and loading the quantized weights from the training phase.
brevitas_model = qfscnn.QFastSCNN(NUM_CLASSES, mode="finn")
brevitas_model = load_state_dict(brevitas_model, path=f"../train_environment/model_weights/quant_params/best_{BIT_WIDTH}_bit_quant_model.pth", strict=False)
brevitas_model.eval();

/home/jose-vitor/finn-repo/deps/brevitas/src/brevitas/__init__.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import DistributionNotFound
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.10/dist-packages/pkg_resources/__init__.py:2871: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packag

Input image shape: torch.Size([1, 3, 1024, 2048])
Input image dtype: torch.uint8
Input mask shape: torch.Size([1, 1024, 2048])
Input mask dtype: torch.uint8
Carregando modelo best_8_bit_quant_model


In [9]:
import torch.nn.functional as F

# Run a foward pass on Brevitas model
with torch.inference_mode():
    brevitas_output = brevitas_model(img_tensor)
brevitas_output_upsampled = F.interpolate(brevitas_output, size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
brevitas_output_mask = torch.softmax(brevitas_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (brevitas_output_mask == smnt_tensor).sum().item()

print(f"Output shape from Brevitas model: {brevitas_output.shape}\n"
      f"Output shape after upsampling: {brevitas_output_upsampled.shape}\n"
      f"Output mask shape: {brevitas_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")

/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/usr/local/lib/python3.10/dist-packages/torch/overrides.py:1528: DeprecationWarning: Defining your `__torch_function__ as a plain method is deprecated and will be an error in future, please define it as a classmethod.
  warnings.warn("Defining your `__torch_function__ as a plain method is deprecated and "
/home/jose-vitor/finn-repo/QFast-SCNN_with_Brevitas_and_FINN/train_environment/models/QFastSCNN.py:71: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  output = torch.c

Output shape from Brevitas model: torch.Size([1, 19, 128, 256])
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 83.38%



In [11]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

model = ModelWrapper(preproc_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

model = model.transform(FoldConstants())
model = model.transform(InferShapes())

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
qonnx_output_upsampled = F.interpolate(torch.from_numpy(qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
qonnx_output_mask = torch.softmax(qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {qonnx_output.shape}\n"
      f"Output shape after upsampling: {qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


Exception: Found unspecified tensor shapes, try infer_shapes

In [8]:
# check output types
print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (brevitas_output_mask == qonnx_output_mask).sum().item()
total_pixels = brevitas_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Brevitas output dtype: torch.float32
QONNX output dtype: float32

Matching pixels: 2090208/2097152 (99.67%)



### Post-Processing

In [9]:
# POST PROC
model = ModelWrapper(preproc_path)

# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())

# Save the postprocessed model
postproc_path = f'./postproc_onnx/{onnx_name}_postproc.onnx'
Path(postproc_path).parent.mkdir(parents=True, exist_ok=True)
model.save(postproc_path)

### Model streamlining

Streamlining transformations listed in the bellow cell has the goal to eliminate floating point operations by moving them around and collapsing them in the previous step transforming them into multi-thresholding nodes.

In [8]:
from finn.transformation.streamline import Streamline
showSrc(Streamline)

class Streamline(Transformation):
    """Apply the streamlining transform, see arXiv:1709.04060."""

    def apply(self, model):
        streamline_transformations = [
            ConvertSubToAdd(),
            ConvertDivToMul(),
            BatchNormToAffine(),
            ConvertSignToThres(),
            MoveMulPastMaxPool(),
            MoveScalarLinearPastInvariants(),
            AbsorbSignBiasIntoMultiThreshold(),
            MoveAddPastMul(),
            MoveScalarAddPastMatMul(),
            MoveAddPastConv(),
            MoveScalarMulPastMatMul(),
            MoveScalarMulPastConv(),
            MoveAddPastMul(),
            CollapseRepeatedAdd(),
            CollapseRepeatedMul(),
            MoveMulPastMaxPool(),
            AbsorbAddIntoMultiThreshold(),
            FactorOutMulSignMagnitude(),
            AbsorbMulIntoMultiThreshold(),
            Absorb1BitMulIntoMatMul(),
            Absorb1BitMulIntoConv(),
            RoundAndClipThresholds(),
        ]
        for tr

In [10]:
from finn.transformation.streamline import Streamline
# we can see the list of apllied transformations here : showSrc(Streamline)
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants
import finn.transformation.streamline.absorb as absorb

model = ModelWrapper(postproc_path)

# STREAMLINE
model = model.transform(Streamline())

# Save the streamlined model
streamlined_path = f'./streamlined_onnx/{onnx_name}_streamlined.onnx'
Path(streamlined_path).parent.mkdir(parents=True, exist_ok=True)
model.save(streamlined_path)

# Print a human readable representation of the graph
#print(helper.printable_graph(model.graph))

In [12]:
showInNetron(streamlined_path)

Serving './streamlined_onnx/quant_model_8_bits_streamlined.onnx' at http://0.0.0.0:8081


After the streamlining process you may notice some Mul and Add blocks included in your model graph on Netron. This blocks are terrible for hardware conversion since they will be converted to float32 operations.

Those blocks must be absorved in MultiThreshold blocks so that they be handled by quantization scale factor and zero point. The FINN streamline stardard transformations, which can be checked one of the previous notebook cells, do not include every use case of all ML models, so the user must look up the [FINN standard model transformations](https://finn.readthedocs.io/en/latest/source_code/finn.transformation.streamline.html#module-finn.transformation.streamline) and use the ones suitable for their model.

Regarding this project, I have used several standard transforms, bus also had to create two new ones, since AveragePool and Concat are blocks not included in the FINN transforms.

In [13]:
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.reorder import MoveMulPastFork, MoveScalarMulPastConv, MoveScalarMulPastConvTranspose, MoveMulPastDWConv, MoveIdenticalOpPastJoinOp, MoveLinearPastEltwiseAdd, MoveScalarLinearPastInvariants, MoveAddPastFork
from finn.transformation.streamline.absorb import AbsorbMulIntoMultiThreshold, AbsorbAddIntoMultiThreshold
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedAdd, CollapseRepeatedMul, CollapseRepeatedOp

import custom_finn_transformations as cft

model = ModelWrapper(streamlined_path)

# running other transformations to clean up the graph as much as possible, to make it ready for the hardware conversion
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model = model.transform(MoveMulPastFork())
model = model.transform(MoveScalarMulPastConvTranspose())
model = model.transform(MoveScalarMulPastConv())
model = model.transform(MoveMulPastDWConv())
model = model.transform(MoveLinearPastEltwiseAdd())
model = model.transform(MoveAddPastFork())
model = model.transform(MoveScalarLinearPastInvariants())
#model = model.transform(cft.MoveMulPastAvgPool())
model = model.transform(cft.MoveScalarLinearPastConcat())
model = model.transform(AbsorbMulIntoMultiThreshold())
model = model.transform(AbsorbAddIntoMultiThreshold())
model = model.transform(CollapseRepeatedAdd())
model = model.transform(CollapseRepeatedMul())
model = model.transform(RoundAndClipThresholds())

# Cleaning up the graph and inferring shapes again, to make it ready for the hardware conversion.
model = model.transform(RemoveUnusedTensors())
model = model.transform(InferShapes())

# Save the streamlined model
ready_for_hw_path = f'./ready_for_hw_conversion_onnx/{onnx_name}_ready_for_hw_conversion.onnx'
Path(ready_for_hw_path).parent.mkdir(parents=True, exist_ok=True)
model.save(ready_for_hw_path)

# Print a human readable representation of the graph
print(helper.printable_graph(model.graph))

/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


graph main_graph (
  %global_in[FLOAT, 1x3x1024x1024]
) initializers (
  %Resize_3_param0[FLOAT, 4]
  %Resize_2_param0[FLOAT, 4]
  %Resize_1_param0[FLOAT, 4]
  %Resize_0_param0[FLOAT, 4]
  %Resize_4_param0[FLOAT, 4]
  %Add_7_param0[FLOAT, 1x19x1x1]
  %Conv_4_param0[FLOAT, 64x48x1x1]
  %Conv_8_param0[FLOAT, 64x384x1x1]
  %Conv_26_param0[FLOAT, 128x576x1x1]
  %Conv_33_param0[FLOAT, 128x1x32x32]
  %Conv_40_param0[FLOAT, 32x128x1x1]
  %Conv_34_param0[FLOAT, 128x1x16x16]
  %Conv_38_param0[FLOAT, 32x128x1x1]
  %Conv_35_param0[FLOAT, 128x1x8x8]
  %Conv_39_param0[FLOAT, 32x128x1x1]
  %Conv_36_param0[FLOAT, 128x1x4x4]
  %Conv_37_param0[FLOAT, 32x128x1x1]
  %Conv_42_param0[FLOAT, 128x1x3x3]
  %Conv_44_param0[FLOAT, 128x1x3x3]
  %Conv_45_param0[FLOAT, 128x128x1x1]
  %Conv_48_param0[FLOAT, 19x128x1x1]
  %Mul_27_param0[FLOAT, scalar]
  %Conv_0_param0[FLOAT, 32x3x3x3]
  %Conv_1_param0[FLOAT, 32x1x3x3]
  %Conv_2_param0[FLOAT, 48x32x1x1]
  %Conv_3_param0[FLOAT, 48x1x3x3]
  %Conv_5_param0[FLOAT, 384x64

In [29]:
showInNetron(ready_for_hw_path)

Serving './ready_for_hw_conversion_onnx/quant_model_8_bits_ready_for_hw_conversion.onnx' at http://0.0.0.0:8081


### Compare the Outputs of Ready for HW Convertion ONNX Model with Outputs of Pre-Proc QONNX Model

Another optional step but highly recommended if the model has unusual layers to FINN framework or if custom QONNX transforms were added to the streamline process (which is the case for my implementation of QFast-SCNN)

In [20]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from qonnx.core.modelwrapper import ModelWrapper
import qonnx.core.onnx_exec as oxe
import onnx.numpy_helper as nph

# add input annotation: UINT8 is what we will feed the model during inference
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

model = ModelWrapper(ready_for_hw_path)
print(f"\nInput datatype: {model.get_tensor_datatype(global_inp_name)}")

# Run the QONNX inference.
input_tensor = img_tensor.detach().numpy().astype(np.float32) # ONNX runtime expects the input tensor to be of type float32, so we convert it to that type before passing it to the model.
input_dict = {"global_in": input_tensor}
output_dict = oxe.execute_onnx(model, input_dict)
hw_qonnx_output = output_dict[list(output_dict.keys())[0]]

# Upsampling the QONNX output to compare to the ground truth mask from the Brevitas model.
hw_qonnx_output_upsampled = F.interpolate(torch.from_numpy(hw_qonnx_output), size=img_tensor.shape[2:], mode='bilinear', align_corners=False)
hw_qonnx_output_mask = torch.softmax(hw_qonnx_output_upsampled, dim=1).argmax(dim=1).to(torch.uint8)

# Calculate how many pixels are matching between the Brevitas output mask and the ground truth mask, and print the results.
matching_pixels = (hw_qonnx_output_mask == smnt_tensor).sum().item()

print(f"Output shape from QONNX model: {hw_qonnx_output.shape}\n"
      f"Output shape after upsampling: {hw_qonnx_output_upsampled.shape}\n"
      f"Output mask shape: {hw_qonnx_output_mask.shape}\n"
      f"Accuracy: {(100 * matching_pixels/smnt_tensor.numel()):.2f}%\n")


Input datatype: UINT8


/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_0_param0 can't be represented with the set datatype annotation (UINT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_14_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_20_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match the datatype annotation.
  warnings.warn(
/home/jose-vitor/finn-repo/deps/qonnx/src/qonnx/util/basic.py:296: UserWarning: The values of tensor MultiThreshold_29_param0 can't be represented with the set datatype annotation (INT8), they will be rounded to match th

Output shape from QONNX model: (1, 19, 128, 256)
Output shape after upsampling: torch.Size([1, 19, 1024, 2048])
Output mask shape: torch.Size([1, 1024, 2048])
Accuracy: 82.85%



In [21]:
# check output types
print(f"Brevitas output dtype: {brevitas_output.dtype}\nQONNX output dtype: {qonnx_output.dtype}\n")

# check if outputs are close enough
matching_pixels = (hw_qonnx_output_mask == qonnx_output_mask).sum().item()
total_pixels = hw_qonnx_output_mask.numel()
print(f"Matching pixels: {matching_pixels}/{total_pixels} ({(100 * matching_pixels/total_pixels):.2f}%)\n")

Brevitas output dtype: torch.float32
QONNX output dtype: float32

Matching pixels: 2087837/2097152 (99.56%)



## Convert to HW

This step does not generate HLS nor RTL code, but rather merges thresholds and matrix vector operations (or convolutions if you use conv nets) into Matrix Vector Activation Units (or Sliding Window Units), if possible, that each represent a layer that FINN can easily work with. Note that an input preprocessing quantiazer will be implemented as a simple standalone Threasholding layer that will then nbe converted to Thresholding_hls layer.

In [3]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from finn.transformation.streamline.absorb import AbsorbConsecutiveTransposes, AbsorbTransposeIntoMultiThreshold, AbsorbTransposeIntoResize, AbsorbConsecutiveTransposes
from finn.transformation.streamline.reorder import MakeScaleResizeNHWC, MoveTransposePastJoinAdd
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
import custom_finn_transformations as cft

# TO HW LAYERS

model = ModelWrapper(ready_for_hw_path)
model = model.transform(LowerConvsToMatMul())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferAddStreamsLayer())
model = model.transform(to_hw.InferChannelwiseLinearLayer())
model = model.transform(to_hw.InferPool())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferVectorVectorActivation())
model = model.transform(RoundAndClipThresholds())
model = model.transform(AbsorbTransposeIntoMultiThreshold())
model = model.transform(to_hw.InferThresholdingLayer())

# Adjustments to the model to make it compatible with the hardware conversion, including reordering and absorbing operations.
model = model.transform(InferShapes())
model = model.transform(InferDataLayouts())
model = model.transform(AbsorbTransposeIntoResize())
#model = model.transform(MakeScaleResizeNHWC())
model = model.transform(to_hw.InferUpsample())
model = model.transform(MoveTransposePastJoinAdd())
model = model.transform(AbsorbConsecutiveTransposes())
model = model.transform(AbsorbTransposeIntoResize())
model = model.transform(AbsorbTransposeIntoMultiThreshold())

# Runs the cuustom transformation to transform Concat nodes from NCHW to NHWC, and them convert them to hardware compatible layers.
model = model.transform(cft.MakeConcatNHWC())
model = model.transform(InferShapes())
model = model.transform(InferDataTypes())
model = model.transform(to_hw.InferConcatLayer())
model = model.transform(RemoveUnusedTensors())

# Save the hw model
hw_layers_path = f'./with_hw_layers/{onnx_name}_hw.onnx'
Path(hw_layers_path).parent.mkdir(parents=True, exist_ok=True)
model.save(hw_layers_path)

#print(helper.printable_graph(model.graph))

/home/jose-vitor/finn-repo/src/finn/transformation/fpgadataflow/convert_to_hw_layers.py:668: UserWarning: Broadcasting Mul(Mul_27)
  warnings.warn("Broadcasting " + str(node.op_type) + "(" + node.name + ")")
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/fmpadding.py:130: UserWarning: inputDataType changing for FMPadding_Batch_: UINT8 -> FLOAT32 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/vectorvectoractivation.py:182: UserWarning: inputDataType changing for VVAU_: UINT8 -> FLOAT32 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/fmpadding.py:130: UserWarning: inputDataType changing for FMPadding_Batch_: FLOAT32 -> UINT8 
  warnings.warn(warn_str)
/home/jose-vitor/finn-repo/src/finn/custom_op/fpgadataflow/vectorvectoractivation.py:182: UserWarning: inputDataType changing for VVAU_: FLOAT32 -> UINT8 
  warnings.warn(warn_str)


In [4]:
showInNetron(hw_layers_path)

Serving './with_hw_layers/quant_model_8_bits_hw.onnx' at http://0.0.0.0:8081


### Isolate HW convertible layers.

In [6]:
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition

model = ModelWrapper(hw_layers_path)
parent_model = model.transform(CreateDataflowPartition())

# Save the dataflow partition model
df_part_path = f'./df_part/{onnx_name}_df_part.onnx'
Path(df_part_path).parent.mkdir(parents=True, exist_ok=True)
parent_model.save(df_part_path)

In [8]:
showInNetron(df_part_path)

Serving './df_part/quant_model_8_bits_df_part.onnx' at http://0.0.0.0:8081
